In [ ]:
import pandas as pd
import numpy as np

from sentence_transformers import SentenceTransformer

# Та же таблица сообщений, что и для признаков переписки.
messages = pd.read_csv("data/data_wthout_anomailes/messages_wthout_anomailes.csv")

chat_ids = messages[['chat_id']].dropna().drop_duplicates()

messages = messages.dropna(subset=["text"]).copy()
messages["text"] = messages["text"].astype(str)

messages

In [ ]:
model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
)
    
display(model.device)

In [ ]:
# POSITIVE — признаки сделки 
# Категория	Что ищем
# purchase_intent	Намерение купить
# seller_confirmation	Подтверждение наличия товара
# price_negotiation	Обсуждение цены
# reservation	Бронирование товара
# meeting	Договорённость о встрече
# delivery	Доставка и отправка
# payment	Обсуждение оплаты
# deal_confirmation	Подтверждение завершения сделки
# item_received	Получение товара
# seller_sold	Сообщение продавца о продаже

# NEGATIVE — признаки несостоявшейся сделки
# Категория	Что ищем
# buyer_refusal	Покупатель отказался
# seller_refusal	Продавец отказался продавать
# price_disagreement	Не договорились о цене
# meeting_failure	Встреча не состоялась
# cancellation	Отмена договорённости
# item_unavailable	Товар больше недоступен
# external_sale	Продажа в другом месте
# seller_changed_mind	Продавец передумал

# NEUTRAL — слабые или фоновые признаки
# Категория	Что ищем
# general_questions	Общие вопросы
# product_condition	Обсуждение состояния товара
# product_details	Характеристики и комплектация
# contact_exchange	Обмен контактами
# availability_question	Вопрос о наличии
# uncertain_intent	Неопределённый интерес

In [ ]:
sale_signals = {

    # ==================================================
    # POSITIVE: ПРИЗНАКИ ПОТЕНЦИАЛЬНОЙ И СОСТОЯВШЕЙСЯ СДЕЛКИ
    # ==================================================

    "purchase_intent": [
        "Я хочу купить этот товар",
        "Готов приобрести ваш товар",
        "Меня интересует покупка",
        "Я заинтересован в приобретении",
        "Хочу забрать этот товар",
        "Готов забрать прямо сегодня",
        "Могу купить прямо сейчас",
        "Я согласен на покупку",
        "Хочу приобрести именно этот товар",
        "Можно оформить покупку?",
        "Я беру этот товар",
        "Давайте оформим сделку",
        "Готов забрать без лишних вопросов",
        "Мне подходит ваше предложение",
        "Я готов приобрести его",
        "Хочу купить, если товар ещё доступен",
        "Договорились, я покупаю",
        "Могу приехать и забрать",
        "Готов забрать за указанную цену",
        "Я точно заинтересован в покупке",
        "Хочу забрать его себе",
        "Покупка меня интересует",
        "Давайте договоримся о покупке",
        "Готов приобрести без торга",
        "Я согласен приобрести товар"
    ],

    "seller_confirmation": [
        "Да, товар ещё продаётся",
        "Да, объявление актуально",
        "Товар в наличии",
        "Да, можете забрать",
        "Товар пока не продан",
        "Ещё не продал, можете приехать",
        "Да, всё доступно",
        "Товар свободен",
        "Можете забрать в любое удобное время",
        "Да, продаю",
        "Объявление актуально, пишите",
        "Товар пока у меня",
        "Да, можно договориться о встрече",
        "Могу продать вам",
        "Товар ещё доступен для покупки",
        "Да, готов продать",
        "Можете приезжать за ним",
        "Пока никто не забрал",
        "Да, товар не забронирован",
        "Готов встретиться для продажи"
    ],

    "price_negotiation": [
        "Можно немного уступить в цене?",
        "Сделаете небольшую скидку?",
        "Готов купить за меньшую сумму",
        "Давайте договоримся по цене",
        "Какая минимальная цена?",
        "Могу предложить другую сумму",
        "Согласны продать дешевле?",
        "Если заберу сегодня, сделаете скидку?",
        "Давайте сойдёмся на этой цене",
        "Предлагаю встретиться посередине",
        "Готов заплатить указанную сумму",
        "Цена меня устраивает",
        "Согласен на вашу цену",
        "Договорились по стоимости",
        "Хорошо, беру за эту сумму",
        "Сколько будет стоить с доставкой?",
        "Могу перевести задаток",
        "За эту цену готов забрать",
        "Давайте зафиксируем стоимость",
        "Хорошо, договорились о цене"
    ],

    "reservation": [
        "Отложите товар для меня",
        "Можно забронировать товар?",
        "Забронируйте, пожалуйста, до вечера",
        "Я точно приеду, придержите товар",
        "Можете пока никому не продавать?",
        "Готов внести предоплату за бронь",
        "Оставьте товар за мной",
        "Давайте договоримся о резерве",
        "Можно забрать завтра, придержите?",
        "Я приеду за забронированным товаром",
        "Снимите объявление, я его забираю",
        "Давайте зафиксируем договорённость",
        "Товар остаётся за мной?",
        "Подержите его до выходных",
        "Я подтверждаю бронь"
    ],

    "meeting": [
        "Когда можно приехать за товаром?",
        "Где можно встретиться?",
        "Давайте договоримся о встрече",
        "Могу подъехать сегодня вечером",
        "Во сколько вам удобно встретиться?",
        "Напишите адрес, куда подъехать",
        "Я буду у вас через час",
        "Давайте встретимся возле метро",
        "Могу приехать завтра утром",
        "Когда вам удобно передать товар?",
        "Давайте согласуем время встречи",
        "Я уже выезжаю к вам",
        "Буду на месте через 15 минут",
        "Могу забрать товар лично",
        "Куда подъехать для покупки?",
        "Давайте встретимся сегодня",
        "Я подъехал, где вас найти?",
        "Встречаемся в оговорённом месте",
        "Подтвердите время встречи",
        "Хорошо, тогда до встречи"
    ],

    "delivery": [
        "Можете отправить товар доставкой?",
        "Отправляете через Авито Доставку?",
        "Давайте оформим доставку",
        "Я готов оплатить доставку",
        "Можете отправить почтой?",
        "Когда сможете отправить товар?",
        "Давайте оформим отправку",
        "Я оформил заказ с доставкой",
        "Отправьте, пожалуйста, после оплаты",
        "Посылка уже отправлена?",
        "Напишите трек-номер отправления",
        "Товар можно доставить курьером?",
        "Давайте согласуем способ доставки",
        "Я готов получить товар через пункт выдачи",
        "Отправка меня устраивает",
        "Хорошо, оформляю доставку",
        "Договорились отправить завтра",
        "Посылку можно забрать в пункте выдачи",
        "Подтвердите отправку товара",
        "Товар уже передан в доставку"
    ],

    "payment": [
        "Куда перевести деньги?",
        "Как можно оплатить товар?",
        "Я готов оплатить покупку",
        "Деньги уже перевёл",
        "Оплата прошла успешно",
        "Перевёл вам указанную сумму",
        "Оплатил заказ",
        "Могу оплатить наличными",
        "Оплачу при получении",
        "Давайте согласуем способ оплаты",
        "Сколько нужно перевести?",
        "Оплата произведена полностью",
        "Вы получили перевод?",
        "Деньги отправлены на вашу карту",
        "Готов рассчитаться при встрече",
        "Я оплатил товар",
        "Получили оплату?",
        "Всё оплачено, спасибо",
        "Давайте завершим оплату",
        "Перевод отправлен, проверьте"
    ],

    "deal_confirmation": [
        "Сделка состоялась, спасибо",
        "Спасибо, договорились",
        "Всё прошло успешно",
        "Спасибо за покупку",
        "Спасибо за продажу",
        "Договорённость выполнена",
        "Товар продан, спасибо",
        "Всё, сделка завершена",
        "Спасибо, всё получил",
        "Покупка успешно состоялась",
        "Мы всё оформили",
        "Деньги получил, спасибо",
        "Спасибо за оперативную сделку",
        "Всё прошло отлично, рекомендую",
        "Договорились, товар теперь мой",
        "Спасибо, сделка завершена",
        "Покупатель забрал товар",
        "Продажа прошла успешно",
        "Всё, вопрос решён",
        "Спасибо, рад покупке"
    ],

    "item_received": [
        "Товар получил, спасибо",
        "Всё забрал, благодарю",
        "Получил посылку",
        "Забрал заказ из пункта выдачи",
        "Товар уже у меня",
        "Всё получил в целости",
        "Посылка пришла, спасибо",
        "Получил покупку, всё отлично",
        "Забрал товар, всё хорошо",
        "Спасибо, товар доставлен",
        "Получение прошло успешно",
        "Забрал сегодня, спасибо",
        "Товар соответствует описанию",
        "Покупкой доволен",
        "Всё на месте, спасибо"
    ],

    "seller_sold": [
        "Я уже продал этот товар",
        "Товар уже продан",
        "Продал его другому покупателю",
        "Покупатель уже забрал товар",
        "Товар больше не продаётся",
        "Уже продано, извините",
        "Продажа состоялась",
        "За товаром уже приезжали",
        "Товар забрали сегодня",
        "Уже нашёл покупателя",
        "Продал, спасибо за интерес",
        "Объявление больше не актуально, товар продан",
        "Вещь уже у нового владельца",
        "Продал через объявление",
        "Покупатель забрал его вчера",
        "Спасибо, товар уже нашёл своего покупателя"
    ],


    # ==================================================
    # NEGATIVE: ПРИЗНАКИ НЕСОСТОЯВШЕЙСЯ СДЕЛКИ
    # ==================================================

    "buyer_refusal": [
        "Я передумал покупать",
        "Больше не интересует",
        "Покупать не буду",
        "Отказываюсь от покупки",
        "Мне больше не нужен этот товар",
        "Я решил не приобретать",
        "Не буду забирать товар",
        "Пока откажусь от предложения",
        "Нашёл другой вариант",
        "Покупка отменяется с моей стороны",
        "Извините, не получится купить",
        "Я больше не заинтересован",
        "Передумал, спасибо",
        "Не готов совершать покупку",
        "Спасибо, но я откажусь",
        "Уже не актуально для меня",
        "Не буду приезжать",
        "Решил пока ничего не покупать",
        "Вынужден отказаться от покупки",
        "Пожалуй, не буду брать"
    ],

    "seller_refusal": [
        "Я не готов продавать",
        "Не хочу продавать за такую цену",
        "Отказываюсь от сделки",
        "Не буду продавать вам товар",
        "Продажа отменяется",
        "Я решил оставить товар себе",
        "Не готов отдавать товар",
        "Не могу продать по этой цене",
        "Извините, продавать передумал",
        "Не получится продать вам",
        "Я не согласен на предложенные условия",
        "Товар продавать больше не планирую",
        "Пока не готов к продаже",
        "Решил не расставаться с вещью",
        "Не буду заключать сделку"
    ],

    "price_disagreement": [
        "Ваша цена меня не устраивает",
        "Не договорились по стоимости",
        "Слишком дорого для меня",
        "За такую сумму покупать не буду",
        "Продавец не согласился на скидку",
        "Мы не сошлись в цене",
        "Не готов платить столько",
        "Цена слишком высокая",
        "Не получается договориться о стоимости",
        "Не согласен с предложенной ценой",
        "Торг не состоялся",
        "Не договорились о скидке",
        "Предложенная сумма слишком маленькая",
        "Продавец не хочет уступать",
        "На таких условиях сделка невозможна"
    ],

    "meeting_failure": [
        "Встреча не состоялась",
        "Не смог приехать за товаром",
        "Продавец не пришёл на встречу",
        "Покупатель не приехал",
        "Договорённость о встрече сорвалась",
        "Не удалось встретиться",
        "Вы не пришли в назначенное время",
        "Я не смогу подъехать",
        "Не получилось забрать товар",
        "Встречу пришлось отменить",
        "Не дождался продавца",
        "Не получилось согласовать время",
        "Встреча переносится на неопределённый срок",
        "Я не приехал за товаром",
        "Передача товара не состоялась"
    ],

    "cancellation": [
        "Давайте отменим договорённость",
        "Я отменяю заказ",
        "Покупка отменена",
        "Договорённость больше не актуальна",
        "Не будем продолжать сделку",
        "Давайте прекратим обсуждение",
        "Отмените, пожалуйста, бронь",
        "Больше не нужно резервировать товар",
        "Сделку пришлось отменить",
        "Я отменил оформление",
        "Не будем ничего оформлять",
        "Заказ отменён",
        "Решил отменить покупку",
        "Договорённость аннулируется",
        "Не получится завершить сделку"
    ],

    "item_unavailable": [
        "Товара больше нет в наличии",
        "Товар уже забрали",
        "Объявление больше не актуально",
        "Этот товар больше не продаётся",
        "Уже нет возможности купить",
        "Товар продан другому человеку",
        "Вещь уже недоступна",
        "К сожалению, товар закончился",
        "Товар больше не у меня",
        "Его уже забрали",
        "Больше не могу предложить этот товар",
        "Товар снят с продажи",
        "Продажа этого товара прекращена",
        "Товар уже передан другому покупателю",
        "Больше не принимаю предложения по этому товару"
    ],

    "external_sale": [
        "Продал товар знакомому",
        "Продал не через Авито",
        "Продал на другой площадке",
        "Продал через социальные сети",
        "Товар забрал мой знакомый",
        "Продал родственнику",
        "Нашёл покупателя самостоятельно",
        "Продал через другой сайт",
        "Договорился с покупателем вне площадки",
        "Продал человеку из своего окружения",
        "Товар купил мой коллега",
        "Продал через знакомых",
        "Покупатель нашёлся не на Авито",
        "Продал через другой сервис",
        "Уже продал вне площадки"
    ],

    "seller_changed_mind": [
        "Передумал продавать",
        "Решил оставить товар себе",
        "Больше не хочу расставаться с вещью",
        "Объявление снимаю, продавать не буду",
        "Пока оставлю товар у себя",
        "Решил не продавать",
        "Товар мне ещё нужен",
        "Продажа больше не актуальна",
        "Передумал выставлять товар на продажу",
        "Пока не планирую продавать",
        "Оставлю вещь для личного использования",
        "Решил отложить продажу",
        "Больше не рассматриваю продажу",
        "Не хочу сейчас продавать эту вещь",
        "Сниму объявление, товар оставлю себе"
    ],


    # ==================================================
    # NEUTRAL: ФОНОВЫЕ ПРИЗНАКИ
    # ==================================================

    "general_questions": [
        "Здравствуйте, у меня есть вопрос",
        "Добрый день, подскажите пожалуйста",
        "Можно задать несколько вопросов?",
        "Вы получили моё сообщение?",
        "Когда сможете ответить?",
        "Спасибо за ответ",
        "Хорошо, понял вас",
        "Можете уточнить информацию?",
        "Я хотел бы узнать подробнее",
        "Есть несколько вопросов по объявлению",
        "Вы сейчас на связи?",
        "Подскажите, пожалуйста",
        "Добрый вечер",
        "Здравствуйте, интересует объявление",
        "Можно с вами связаться?"
    ],

    "product_condition": [
        "В каком состоянии находится товар?",
        "Есть ли какие-нибудь дефекты?",
        "Товар полностью исправен?",
        "Есть ли повреждения?",
        "Сколько времени использовалась вещь?",
        "Есть ли царапины или сколы?",
        "Товар новый или бывший в употреблении?",
        "Работает ли устройство?",
        "Есть ли следы использования?",
        "Можно посмотреть фотографии дефектов?",
        "Всё ли работает исправно?",
        "Состояние соответствует фотографиям?",
        "Были ли ремонты?",
        "Есть ли гарантия?",
        "Товар требует ремонта?"
    ],

    "product_details": [
        "Какая модель товара?",
        "Какие характеристики у устройства?",
        "Что входит в комплект?",
        "Какой размер товара?",
        "Какого года выпуска устройство?",
        "Есть ли оригинальная упаковка?",
        "Можете прислать дополнительные фотографии?",
        "Какой цвет у товара?",
        "Есть ли документы?",
        "Какая комплектация?",
        "Какой срок использования?",
        "Можете рассказать подробнее?",
        "Есть ли инструкция?",
        "Какой производитель?",
        "Какие размеры у изделия?"
    ],

    "contact_exchange": [
        "Давайте созвонимся",
        "Можно ваш номер телефона?",
        "Напишите мне в мессенджер",
        "Оставьте контакт для связи",
        "Как с вами связаться?",
        "Я отправил свой номер",
        "Давайте обсудим по телефону",
        "Можете позвонить мне?",
        "Напишите, когда будете свободны",
        "Я доступен для звонка"
    ],

    "availability_question": [
        "Товар ещё актуален?",
        "Объявление ещё действует?",
        "Ещё продаёте?",
        "Товар всё ещё доступен?",
        "Его уже купили?",
        "Можно узнать, актуально ли объявление?",
        "Вы ещё продаёте эту вещь?",
        "Товар пока в продаже?",
        "Можно ли его приобрести?",
        "Объявление не закрыто?"
    ],

    "uncertain_intent": [
        "Я пока думаю над покупкой",
        "Возможно, заинтересуюсь позже",
        "Пока рассматриваю варианты",
        "Мне нужно подумать",
        "Я ещё не определился",
        "Пока не могу дать точный ответ",
        "Возможно, вернусь к вам позже",
        "Сейчас просто изучаю предложения",
        "Пока сравниваю цены",
        "Ещё рассматриваю ваше предложение",
        "Мне нужно посоветоваться",
        "Подумаю и напишу позже",
        "Пока не готов принять решение",
        "Не знаю, получится ли забрать",
        "Пока просто интересуюсь"
    ]
}

signal_weights = {
    # POSITIVE — признаки в пользу сделки
    "purchase_intent": 0.8,
    "seller_confirmation": 1.2,
    "price_negotiation": 0.5,
    "reservation": 0.7,
    "meeting": 0.6,
    "delivery": 1.0,
    "payment": 1.3,
    "deal_confirmation": 2.0,
    "item_received": 2.0,
    "seller_sold": 1.8,

    # NEGATIVE — признаки против сделки
    "buyer_refusal": -1.0,
    "seller_refusal": -1.2,
    "price_disagreement": -0.8,
    "meeting_failure": -1.2,
    "cancellation": -1.0,
    "item_unavailable": -0.7,
    "external_sale": -2.0,
    "seller_changed_mind": -1.5,

    # NEUTRAL — контекстные признаки
    "general_questions": 0.0,
    "product_condition": 0.0,
    "product_details": 0.0,
    "contact_exchange": 0.1,
    "availability_question": 0.0,
    "uncertain_intent": -0.2
}

signal_embeddings = {}

for category, phrases in sale_signals.items():
    signal_embeddings[category] = model.encode(
        phrases,
        normalize_embeddings=True,
        show_progress_bar=True
    )

if messages.empty:
    for category in signal_weights:
        messages[category] = pd.Series(index=messages.index, dtype=float)
else:
    # Одинаковые тексты кодируем один раз.
    text_codes, unique_texts = pd.factorize(messages["text"])
    message_embeddings = model.encode(
        unique_texts.tolist(),
        normalize_embeddings=True,
        batch_size=128,
        show_progress_bar=True
    )

    for category, embeddings in signal_embeddings.items():

        similarities = message_embeddings @ embeddings.T

        messages[category] = similarities.max(axis=1)[text_codes]

categories = list(signal_weights.keys())

# Умножаем оценки сходства на веса
weighted_scores = messages[categories].mul(signal_weights)

# Складываем веса для каждого сообщения
messages["deal_signal_score"] = weighted_scores.sum(axis=1)

# Веса пяти последних текстовых сообщений каждого чата.
recent_message_weights = {0: 3.0, 1: 2.5, 2: 2.0, 3: 1.75, 4: 1.5}
message_order = messages[["chat_id"]].assign(
    sent_at=pd.to_datetime(messages["sent_at"], errors="coerce")
)
# При одинаковом времени более поздней считаем строку ниже в исходном CSV.
message_order = (
    message_order.dropna(subset=["sent_at"])
    .sort_index(ascending=False)
    .sort_values("sent_at", ascending=False, kind="stable")
)
recent_rank = message_order.groupby("chat_id", sort=False).cumcount()
messages["recency_weight"] = recent_rank.map(recent_message_weights).reindex(messages.index).fillna(1.0)
messages["weighted_deal_signal_score"] = messages["deal_signal_score"] * messages["recency_weight"]

messages = messages.sort_values(by="weighted_deal_signal_score", ascending=False)
messages

## Сигналы сделки по чатам

В `chat_scores` одна строка на `chat_id`. `deal_signal_score` — разность между самым сильным положительным сигналом и модулем самого сильного отрицательного сигнала по сообщениям чата. Повторение сообщения не увеличивает оценку.

Для отбора чатов на проверку таблица отсортирована по `deal_signal_strength` — силе большего из двух сигналов. Поэтому вверху могут оказаться как подтверждения, так и отказы. Если оба сигнала сильные, их разность может быть близка к нулю, поэтому обе оценки сохранены отдельно. Чаты без оценённых сообщений имеют пропуски в оценках. Это эвристика для анализа переписки.

In [ ]:
weighted_message_score = messages["weighted_deal_signal_score"]

chat_evidence = (
    messages.assign(
        strongest_positive_signal=weighted_message_score.clip(lower=0),
        strongest_negative_signal=(-weighted_message_score).clip(lower=0),
    )
    .groupby("chat_id", as_index=False)
    .agg(
        scored_message_count=("deal_signal_score", "count"),
        strongest_positive_signal=("strongest_positive_signal", "max"),
        strongest_negative_signal=("strongest_negative_signal", "max"),
    )
)

chat_scores = chat_ids.merge(chat_evidence, on="chat_id", how="left", validate="one_to_one")
chat_scores["scored_message_count"] = chat_scores["scored_message_count"].fillna(0).astype(int)
chat_scores["deal_signal_score"] = (
    chat_scores["strongest_positive_signal"] - chat_scores["strongest_negative_signal"]
)
chat_scores["deal_signal_strength"] = chat_scores[
    ["strongest_positive_signal", "strongest_negative_signal"]
].max(axis=1)
chat_scores = chat_scores.sort_values(
    "deal_signal_strength", ascending=False, kind="stable"
).reset_index(drop=True)

chat_scores = chat_scores[chat_scores["scored_message_count"] > 0]

chat_scores

In [ ]:
listing_id_by_chat = messages[["chat_id", "listing_id"]].drop_duplicates().set_index("chat_id")["listing_id"]
chat_scores["listing_id"] = chat_scores["chat_id"].map(listing_id_by_chat)
assert chat_scores["listing_id"].notna().all()

chat_scores.drop(columns="chat_id").to_csv("chat_features.csv", index=False)
chat_scores